# Domain 4: Prompt Engineering & Structured Output
## TS 4.5: Design Efficient Batch Processing Strategies with Message Batches API
**Certification Focus**: CCAR-F Domain 4 (20% Exam Weight) | **Scenario Alignment**: Scenario 6 (Structured Data Extraction & Auditing)

---

### 🎯 Key Architectural Principles
1. **The Message Batches API**:
   - **50% Cost Discount** on all input and output tokens compared to standard on-demand pricing.
   - **24-Hour SLA**: Results processed asynchronously within a 24-hour turnaround window.
   - **Non-Real-Time Workloads**: Ideal for offline backfills, nightly vulnerability scans, and large document archive migrations.
2. **Partial Failure Resubmission**: Batches may have individual failures (e.g. content policy, timeouts). The engine must filter succeeded vs failed requests by `custom_id` and resubmit **only** the failed subset.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 4.5 Message Batches API.")

### 📦 Step 2: Message Batches Engine Implementation
We implement the batch management engine with partial failure isolation and cost savings calculations.

In [ ]:
# Step 2: Batch Engine Logic
@dataclass
class BatchRequest:
    custom_id: str
    params: Dict[str, Any]


@dataclass
class BatchResponse:
    batch_id: str
    status: str
    total_requests: int
    succeeded: List[str]
    failed: List[str]


class MessageBatchesEngine:
    def __init__(self) -> None:
        self.batches: Dict[str, BatchResponse] = {}

    def submit_batch(self, requests: List[BatchRequest]) -> str:
        batch_id = f"batch_{len(self.batches) + 1:03d}"
        succeeded = []
        failed = []

        # Simulate batch processing
        for req in requests:
            # Simulate failure for requests over token limit
            if req.params.get("token_estimate", 0) > 100_000:
                failed.append(req.custom_id)
            else:
                succeeded.append(req.custom_id)

        self.batches[batch_id] = BatchResponse(
            batch_id=batch_id,
            status="ended",
            total_requests=len(requests),
            succeeded=succeeded,
            failed=failed,
        )
        return batch_id

    def filter_failed_for_resubmission(self, batch_id: str) -> List[str]:
        """Extracts failed custom_ids to prevent re-running already successful requests."""
        batch = self.batches.get(batch_id)
        return batch.failed if batch else []

    @staticmethod
    def calculate_cost_savings(total_tokens: int, base_cost_per_million: float = 3.0) -> Dict[str, float]:
        """Message Batches API provides an exact 50% discount."""
        standard_cost = (total_tokens / 1_000_000) * base_cost_per_million
        batch_cost = standard_cost * 0.50
        savings = standard_cost - batch_cost
        return {
            "standard_cost": round(standard_cost, 4),
            "batch_cost": round(batch_cost, 4),
            "savings": round(savings, 4),
            "discount_pct": 50.0,
        }

print(" MessageBatchesEngine initialized.")

### 🧪 Step 3: Interactive Simulation
1. Submit batch of requests.
2. Filter failed custom_ids for partial resubmission.
3. Calculate cost savings on 50M tokens.

In [ ]:
# Step 3: Simulation
engine = MessageBatchesEngine()

# 1. Submit batch with mixed token estimates
requests = [
    BatchRequest(custom_id="doc_001", params={"token_estimate": 2500}),
    BatchRequest(custom_id="doc_002", params={"token_estimate": 150_000}),  # Fails!
    BatchRequest(custom_id="doc_003", params={"token_estimate": 4000}),
]
batch_id = engine.submit_batch(requests)
print(f"Batch submitted: {batch_id}")

# 2. Extract failed items for resubmission
failed_ids = engine.filter_failed_for_resubmission(batch_id)
print("Failed requests needing resubmission:", failed_ids)

# 3. Calculate 50% cost savings on 50M tokens
savings = MessageBatchesEngine.calculate_cost_savings(50_000_000)
print("\nCost Analysis on 50M Tokens:", savings)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Succeeded and failed custom_ids are partitioned.
2. Resubmission selects only failed IDs.
3. Batch cost savings calculate to exactly 50%.

In [ ]:
# Step 4: Verification Checks
assert failed_ids == ["doc_002"]
assert savings["discount_pct"] == 50.0
assert savings["batch_cost"] == savings["standard_cost"] / 2

print(" All verification assertions passed! Message Batches API verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use the **Message Batches API** for non-real-time jobs to capture the **50% cost reduction**.
- **Rule 2**: The turnaround SLA is **24 hours**.
- **Rule 3**: On partial batch failure, resubmit **only the failed custom_ids**; never re-run the entire batch.